## **Phase 2 (Peer Review)**

#### Project Information

**Research Question:** How can regional and weather factors like cloud cover and elevation influence the performance of drivers across teams, possibly highlighting team-specific specializations?

**Dataset Description (F1 Race Data)**
> https://openf1.org/docs/

<sup><sub>**Notes about the dataset:** The first dataset includes historical F1 race data able to be filtered by driver, race, or date. We can compare different teams and individuals and how their lap times, their pit duration, and any other relevant podium-seeking information varies. Via the OpenF1 API (https://github.com/br-g/openf1), we get access to car telemetry (https://openf1.org/docs/#car-data), lap times (https://openf1.org/docs/#laps), race positions (https://openf1.org/docs/#position), pit stops (https://openf1.org/docs/#pit), and race control (https://openf1.org/docs/#race-control) from all races, teams, and racers. Access is limited to historical data, meaning the 2023 season onward. The weather information (https://openf1.org/docs/#weather) is quite limited excluding factors like cloud cover and elevation, which is why we are looking at Open Meteo API to supplement OpenF1. On OpenF1, there are also generous rate limits of 3 requests per second or 30 requests per minute (https://openf1.org/#faq). OpenF1 is a free, community-run API, so its data is generally reliable for stats because it's a large community, but it isn't an official source of record. Veritably, OpenF1 is directly based on F1 timing/telemetry data making it suitable for academic-style preparatory work, which is our purpose.</sub></sup>

**Dataset Description (Weather Data)**
> https://open-meteo.com/en/docs/historical-weather-api

<sup><sub>**Notes about the dataset:** The second dataset includes realtime, historical weather data that can be used to supplement our analysis of race results. The main access limitation with the Open Meteo API is the spatial resolution, having grid cells ranging from 11 km to 25 km in size. The specific set we’d end up using is likely the ERA5 (https://doi.org/10.24381/cds.adbb2d47) for our use case. These reports are hourly and constructed from data 1950 and onward. Open Meteo, construction described at this site (https://doi.org/10.5281/ZENODO.7970649) may lose some reliability in how its data is built off of reanalysis of various sources (weather stations, aircrafts, buoys, radar, etc.) rather than a point observation.</sub></sup>

#### Data cleaning

##### Libraries

In [39]:
# installing necessary libraries
%pip install seaborn pandas duckdb requests numpy

Note: you may need to restart the kernel to use updated packages.


In [40]:
# loading necessary libraries and packages to handle our data
import requests # to extract from datasets
import pandas as pd # to handle the dataframes
import duckdb # to clean the data
from datetime import datetime # to handle the date time format

Note for above code chunk(s), will be modifying and editing as we go to ensure we have what we need to handle our data.

##### Data Import and Cleansing

In [41]:
# code OpenF1 API informs us to use to import
from urllib.request import urlopen
import json
response = urlopen('https://api.openf1.org/v1/sessions')
data = json.loads(response.read().decode('utf-8'))
temp_races_df = pd.DataFrame(data) # last line of provided code from API

For above code, this was provided by OpenF1 API in (https://openf1.org/docs) for information on how to import/scrape.

In [42]:
temp_races_df # checking the column headers
temp_races_df['session_type'].unique() # checking to see what the different types are to make sure we don't truncate

# selecting the wanted columns and races
races_df = duckdb.sql("SELECT session_type, session_name, date_start, date_end, circuit_short_name, country_name, location, year \
                      FROM temp_races_df WHERE ((session_type = 'Race' OR session_type = 'Qualifying') AND is_cancelled = False) \
                      AND year < 2026").df()
races_df # checking the product and modifying sql as necessary to prune canceled races and races from 2026 (ongoing) season
races_df['date_start'] = pd.to_datetime(races_df['date_start'], format='mixed') # reformat dates
races_df['date_end'] = pd.to_datetime(races_df['date_end'], format='mixed')
# wanting to uniform the time zone and make it more easily comprehensible for human reading
races_df['date_start'] = races_df['date_start'].dt.tz_convert("EST").dt.tz_localize(None)
races_df['date_end'] = races_df['date_end'].dt.tz_convert("EST").dt.tz_localize(None)
races_df # checking date time

,session_type,session_name,date_start,date_end,circuit_short_name,country_name,location,year
0,Qualifying,Qualifying,2023-03-04 10:00:00,2023-03-04 11:00:00,Sakhir,Bahrain,Sakhir,2023
1,Race,Race,2023-03-05 10:00:00,2023-03-05 12:00:00,Sakhir,Bahrain,Sakhir,2023
2,Qualifying,Qualifying,2023-03-18 12:00:00,2023-03-18 13:00:00,Jeddah,Saudi Arabia,Jeddah,2023
3,Race,Race,2023-03-19 12:00:00,2023-03-19 14:00:00,Jeddah,Saudi Arabia,Jeddah,2023
4,Qualifying,Qualifying,2023-04-01 01:00:00,2023-04-01 02:00:00,Melbourne,Australia,Melbourne,2023
...,...,...,...,...,...,...,...,...
171,Race,Sprint,2025-11-29 09:00:00,2025-11-29 10:00:00,Lusail,Qatar,Lusail,2025
172,Qualifying,Qualifying,2025-11-29 13:00:00,2025-11-29 14:00:00,Lusail,Qatar,Lusail,2025
173,Race,Race,2025-11-30 11:00:00,2025-11-30 13:00:00,Lusail,Qatar,Lusail,2025
174,Qualifying,Qualifying,2025-12-06 09:00:00,2025-12-06 10:00:00,Yas Marina Circuit,United Arab Emirates,Yas Island,2025


Here, we have cleaned a DataFrame to contain the dates and times for races and the races that qualify the teams to participate in the larger race for narrowing down the time and date for the weather conditions. We also made sure to maintain the specific names of the locations to further assist us when looping in information from Open Meteo. We found the session types listed were [Practice, Qualifying, Race] and chose to disregard Practices since teams use these for practice and performance may not be up to par; in other words, since the teams are warming up or practicing on the track, the information consumed may not be as relevant to driver overall performance when representing their team name. One other important thing to note is that we filtered out the race dates from 2026. Since the 2026 F1 season is still ongoing, these races will lack information about the performance of teams and their drivers. We prefer to handle a constantly changing dataset, so we decided to stick with 2023, 2024, and 2025 where all the data is set.